In [ ]:
# ============================================================
# CELL 1 — Imports, configuration, and path setup
# ============================================================
# WTI & Brent Price Monitor
# Sources: yfinance (futures) + EIA API (spot cross-reference)
# ─────────────────────────────────────────────────────────────

# Standard library
import sys
import os
import glob
import datetime as dt
from pathlib import Path

# Data
import pandas as pd
import numpy as np

# Market data
import yfinance as yf

# HTTP
import requests

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

# Display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

# ── Path setup ───────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import config
import chart_style as cs

cs.apply_style()          # must run before any figure is created

API_KEY   = config.EIA_API_KEY
BASE_URL  = "https://api.eia.gov/v2"
CHART_DIR = PROJECT_ROOT / "charts"
DATA_DIR  = PROJECT_ROOT / "data"

# ── Price data configuration ─────────────────────────────────
# Both tickers track front-month by delivery month.
# NOTE: Brent futures expire ~3 weeks before the WTI contract
# for the same delivery month, so continuous series elsewhere
# (TradingView BRN1!, Google) may roll Brent forward while WTI
# is still on the prior month. Our spread is same-delivery-month,
# which strips out calendar-spread contamination. Expect it to
# read WIDER than quoted "front-month" spreads during roll weeks.
WTI_TICKER   = "CL=F"    # front-month WTI  (currently CLX26)
BRENT_TICKER = "BZ=F"    # front-month Brent (currently BZX26)

# EIA spot price series (weekly, used as cross-reference)
EIA_WTI_ROUTE = "/petroleum/pri/spt/data/"

# ── Date windows ─────────────────────────────────────────────
PRICE_START = "2007-01-01"   # Brent data quality starts ~1987
END_DATE    = dt.date.today()

SPREAD_WINDOW_DAYS = 252     # ~1 trading year for rolling stats
RECENT_DAYS        = 365 * 3 # 3 years for main price chart focus

# Percentile rank window — deliberately NOT the full history.
# The US crude export ban was lifted in December 2015. Before
# that, landlocked WTI could disconnect from waterborne Brent in
# ways the current market structure doesn't allow, so ranking
# today's spread against pre-2016 observations compares two
# different regimes. 2016+ keeps the comparison honest.
SPREAD_PCTILE_START = "2016-01-01"

# ── Event markers ────────────────────────────────────────────
# Dated explicitly rather than detected from a price threshold.
# A heuristic like "first day WTI crossed $75" silently moves as
# the data changes; a date doesn't.
EVENTS = {
    pd.Timestamp("2026-02-28"): "Hormuz closed\nUS+Israel strike Iran",
}

# ── Cache settings ───────────────────────────────────────────
WTI_CACHE   = DATA_DIR / "wti_price_cache.csv"
BRENT_CACHE = DATA_DIR / "brent_price_cache.csv"

# ── Sanity check ─────────────────────────────────────────────
print("=" * 58)
print("  WTI & Brent Price Monitor — configuration loaded")
print("=" * 58)
print(f"  Project root  : {PROJECT_ROOT}")
print(f"  API key       : {'loaded' if API_KEY else 'MISSING'}")
print(f"  WTI ticker    : {WTI_TICKER}")
print(f"  Brent ticker  : {BRENT_TICKER}")
print(f"  Price history : {PRICE_START} → {END_DATE}")
print(f"  Spread window : {SPREAD_WINDOW_DAYS} trading days")
print(f"  Pctile from   : {SPREAD_PCTILE_START}")
print(f"  Cache dir     : {DATA_DIR}")
print("=" * 58)

In [ ]:
# ============================================================
# CELL 2 — Caching price fetch function
# ============================================================
# Why caching matters here:
#  yfinance downloading 35+ years of daily price data takes
#  5-15 seconds and hammers Yahoo's servers on every run.
#  Instead we: download once -> save to CSV -> on future runs
#  load from CSV and only fetch data newer than the last 
#  cached date. Refresh time drops to under 2 seconds!

# This pattern (cache + incremental update) is standard in
# any data pipeline that pulls from a rate-limited source.
# ─────────────────────────────────────────────────────────────

def fetch_price_with_cache(
    ticker: str,
    cache_path: Path,
    start: str,
    end: dt.date,
    price_col: str = "Close",
) -> (pd.DataFrame, bool):
    """
    Fetch daily closing prices for a ticker via yfinance,
    using a local CSV cache for speed on repeat runs.

    Parameters
    ----------
    ticker     : yfinance ticker string e.g. 'CL=F'
    cache_path : Path to local CSV cache file
    start      : earliest date to fetch 'YYYY-MM-DD'
    end        : latest date (today usually)
    price_col  : which OHLCV column to keep (default Close)

    Returns
    -------
    pd.DataFrame with columns: date, price
    bool: True/False depending on whether data was downloaded for the ticker
    Daily frequency, sorted ascending, NaNs dropped.
    """
    end_str = end.strftime("%Y-%m-%d")

    # ── Attempt to load from cache ───────────────────────────
    if cache_path.exists():
        df_cache = pd.read_csv(cache_path, parse_dates=["date"])
        df_cache = df_cache.sort_values("date").reset_index(drop=True)

        last_cached = df_cache["date"].max().date()
        days_stale = (end - last_cached).days

        if days_stale <= 1:
            # Cache is current - no current fetch needed
            # <=1 check accounts for weekends/holidays - no need to call the API on Saturday/Sunday/holiday
            print(f" [{ticker}] Cache current "
                  f"(last: {last_cached}) - skipping download")
            return (df_cache, False)
        
        # Cache exists but needs topping up
        # Fetch only from day after last cached date
        fetch_start = (last_cached + dt.timedelta(days=1)).strftime("%Y-%m-%d")
        print(f" [{ticker}] Cache stale by {days_stale} days - "
              f"fetching {fetch_start} -> {end_str}")
        is_incremental = True

    else:
        # No cache - full download
        fetch_start = start
        df_cache = None
        is_incremental = False
        print(f" [{ticker}] No cache found - "
              f"full download {fetch_start} -> {end_str}")
    # ── Download from yfinance ───────────────────────────────
    try:
        raw = yf.download(
            ticker,
            start = fetch_start,   
            end = end_str,
            interval = "1d", 
            auto_adjust = True,
            progress = False, # suppress yfinance progress bar
        )
    except Exception as e:
        raise RuntimeError(f"yfinance download failed for {ticker}: {e}")
    
    if raw.empty:
        if is_incremental and df_cache is not None:
            print(f" [{ticker}] No new data available - "
                  f"using cached data")
            return (df_cache, False)
        raise ValueError(
            f"yfinance returned no data for {ticker} "
            f"({fetch_start} -> {end_str})"
        )
    # ── Parse and clean new data ─────────────────────────────
    # yfinance returns a MultiIndex column if only one ticker
    # flatten it cleanly regardless
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    
    df_new = raw[[price_col]].copy()
    df_new.index = pd.to_datetime(df_new.index)
    df_new = df_new.reset_index()
    df_new.columns = ["date", "price"]
    dates = pd.to_datetime(df_new["date"])
    if dates.dt.tz is not None:
        df_new["date"] = dates.dt.tz_convert(None)  # tz-aware -> strip
    else:
        df_new["date"] = dates.dt.tz_localize(None)  # tz-naive -> confirm naive
    df_new = df_new.dropna(subset=["price"])
    df_new = df_new[df_new["price"] != 0] # drop any zero-price artifacts

    # ── Merge with cache if incremental ─────────────────────
    if is_incremental and df_cache is not None:
        df_combined = pd.concat([df_cache, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(
            subset=["date"], keep="last"
        )
    else:
        df_combined = df_new
    
    df_combined = df_combined.sort_values("date").reset_index(drop=True)

    # ── Save updated cache ─────────────────────────────────── 
    df_combined.to_csv(cache_path, index=False)
    print(f"  [{ticker}] Saved {len(df_combined):,} rows → "
          f"{cache_path.name}")
    
    return (df_combined, True)

# ── Smoke test ───────────────────────────────────────────────
# ── Smoke test ───────────────────────────────────────────────
print("=" * 58)
print("  Cell 2 loaded — fetch_price_with_cache() ready")
print("=" * 58)
print(f"  WTI cache  : {WTI_CACHE.name}")
print(f"  Brent cache: {BRENT_CACHE.name}")
print()
print("  Caching logic:")
print("  ┌─ Cache missing  → full download from PRICE_START")
print("  ├─ Cache stale    → incremental fetch (new days only)")
print("  └─ Cache current  → load from disk, skip download")
print("=" * 58)

In [ ]:
# ============================================================
# CELL 3 — Download WTI and Brent price history
# ============================================================
# First run  : downloads full history back to PRICE_START
#              and saves to CSV cache — takes ~10-20 seconds
# Subsequent : loads from cache + fetches only new days
#              — takes ~2 seconds
# ─────────────────────────────────────────────────────────────

print("Fetching price data...")
print()

# ── Download ─────────────────────────────────────────────────
df_wti_daily,   wti_downloaded   = fetch_price_with_cache(
    ticker=WTI_TICKER, cache_path=WTI_CACHE,
    start=PRICE_START, end=END_DATE,
)
df_brent_daily, brent_downloaded = fetch_price_with_cache(
    ticker=BRENT_TICKER, cache_path=BRENT_CACHE,
    start=PRICE_START, end=END_DATE,
)

# ── Merge into single daily DataFrame ───────────────────────
# Outer merge keeps WTI's longer history and every session either
# exchange traded. NYMEX and ICE observe different holidays, so
# the union has gaps in one column or the other.
df_daily = pd.merge(
    df_wti_daily.rename(columns={"price": "wti"}),
    df_brent_daily.rename(columns={"price": "brent"}),
    on="date", how="outer",
)
df_daily = df_daily.dropna(subset=["wti", "brent"], how="all")
df_daily = df_daily.sort_values("date").reset_index(drop=True)

# ── Record what was actually observed, BEFORE any filling ────
# This flag is the whole point of the next block. Once we
# forward-fill, we can no longer tell a real print from a
# carried-forward one — so we capture it first.
both_observed = df_daily["wti"].notna() & df_daily["brent"].notna()

# ── Spread: only where both legs genuinely traded ────────────
# Brent - WTI = the "Brent premium"
#   Positive = Brent more expensive than WTI (normal)
#   Negative = WTI more expensive than Brent (unusual)
#
# A spread is a relationship between two SIMULTANEOUS prices.
# On a day only NYMEX traded, forward-filling Brent would give
# us today's WTI against yesterday's Brent — a number that
# looks like a spread but isn't one. We leave those NaN.
df_daily["spread"] = np.where(
    both_observed,
    df_daily["brent"] - df_daily["wti"],
    np.nan,
)

n_total    = len(df_daily)
n_observed = int(both_observed.sum())
n_gap      = n_total - n_observed

# ── Forward-fill prices for display continuity only ─────────
# Done AFTER the spread, so no filled value can leak into it.
df_daily["wti"]   = df_daily["wti"].ffill()
df_daily["brent"] = df_daily["brent"].ffill()

# ── Resample to weekly (Friday close) ────────────────────────
# Aligns with EIA weekly inventory (released Wednesday, covering
# the week ending Friday). .last() skips NaN, so the weekly
# spread is the last day that week on which BOTH legs traded.
df_weekly = (
    df_daily.set_index("date")[["wti", "brent", "spread"]]
    .resample("W-FRI")
    .last()
    .dropna(subset=["wti", "brent"])
    .reset_index()
)

# ── Rolling spread statistics ────────────────────────────────
# min_periods counts non-NaN observations, so the gaps above
# are handled correctly without any special casing.
df_daily["spread_roll_mean"] = (
    df_daily["spread"].rolling(SPREAD_WINDOW_DAYS, min_periods=60).mean()
)
df_daily["spread_roll_std"] = (
    df_daily["spread"].rolling(SPREAD_WINDOW_DAYS, min_periods=60).std()
)
df_daily["spread_z"] = (
    (df_daily["spread"] - df_daily["spread_roll_mean"])
    / df_daily["spread_roll_std"]
)

# ── 52-week rolling high/low ─────────────────────────────────
# Computed once, here. (Cell 4 used to recompute these with a
# different min_periods and silently overwrite them.)
df_daily["wti_52w_high"]   = df_daily["wti"].rolling(252, min_periods=60).max()
df_daily["wti_52w_low"]    = df_daily["wti"].rolling(252, min_periods=60).min()
df_daily["brent_52w_high"] = df_daily["brent"].rolling(252, min_periods=60).max()
df_daily["brent_52w_low"]  = df_daily["brent"].rolling(252, min_periods=60).min()

# ── Contract alignment check (price-based, not metadata) ─────
# underlyingExchangeSymbol is unreliable for BZ=F — verify by
# matching the continuous price against explicit contracts.
# MAINTENANCE: refresh this candidate list quarterly.
def identify_contract(cont_ticker, candidates, tol=0.05):
    cont = yf.download(cont_ticker, period="5d",
                       auto_adjust=True, progress=False)
    if isinstance(cont.columns, pd.MultiIndex):
        cont.columns = cont.columns.get_level_values(0)
    target = float(cont["Close"].dropna().iloc[-1])
    for c in candidates:
        try:
            r = yf.download(c, period="5d",
                            auto_adjust=True, progress=False)
            if r.empty:
                continue
            if isinstance(r.columns, pd.MultiIndex):
                r.columns = r.columns.get_level_values(0)
            if abs(float(r["Close"].dropna().iloc[-1]) - target) < tol:
                return c, target
        except Exception:
            continue
    return "unidentified", target

cache_refreshed = (wti_downloaded or brent_downloaded)
wti_c = brent_c = None
wti_p = brent_p = None

if cache_refreshed:
    wti_c, wti_p = identify_contract(
        "CL=F", ["CLX26.NYM", "CLZ26.NYM", "CLF27.NYM"])
    brent_c, brent_p = identify_contract(
        "BZ=F", ["BZX26.NYM", "BZZ26.NYM", "BZF27.NYM"])
    print(f"  WTI   CL=F -> {wti_c}   (${wti_p:.2f})")
    print(f"  Brent BZ=F -> {brent_c}   (${brent_p:.2f})")
else:
    print("  Contract check skipped (cache current)")

if (wti_c and brent_c
        and "unidentified" not in (wti_c, brent_c)
        and wti_c[2:5] != brent_c[2:5]):
    print()
    print("  NOTE: legs are on different delivery months —")
    print("  spread includes a calendar component this week")

# ── Summary stats ────────────────────────────────────────────
latest      = df_daily.iloc[-1]
latest_date = df_daily["date"].max().strftime("%B %d, %Y")

wti_52w_high   = df_daily.tail(252)["wti"].max()
wti_52w_low    = df_daily.tail(252)["wti"].min()
brent_52w_high = df_daily.tail(252)["brent"].max()
brent_52w_low  = df_daily.tail(252)["brent"].min()

# ── Spread percentile — post-export-ban regime only ──────────
# See SPREAD_PCTILE_START in Cell 1 for why this isn't full history.
df_pctile = df_daily[
    (df_daily["date"] >= SPREAD_PCTILE_START)
    & df_daily["spread"].notna()
]
latest_spread_obs = df_daily["spread"].dropna().iloc[-1]

spread_regime_mean = df_pctile["spread"].mean()
spread_pctile = (
    (df_pctile["spread"] < latest_spread_obs).sum()
    / len(df_pctile) * 100
)

print()
print("=" * 58)
print("  Price data loaded")
print("=" * 58)
print(f"  Daily rows      : {n_total:,} trading days")
print(f"  Both legs traded: {n_observed:,} days "
      f"({n_gap:,} one-sided, spread left blank)")
print(f"  Weekly rows     : {len(df_weekly):,} weeks")
print(f"  Date range      : {df_daily['date'].min().date()} → "
      f"{df_daily['date'].max().date()}")
print()
print(f"  WTI  (latest)   : ${latest['wti']:.2f}/bbl")
print(f"  WTI  52w range  : ${wti_52w_low:.2f} – ${wti_52w_high:.2f}")
print()
print(f"  Brent (latest)  : ${latest['brent']:.2f}/bbl")
print(f"  Brent 52w range : ${brent_52w_low:.2f} – ${brent_52w_high:.2f}")
print()
print(f"  Spread (Brent–WTI) : ${latest_spread_obs:.2f}/bbl")
print(f"  Spread mean (2016+): ${spread_regime_mean:.2f}/bbl")
print(f"  Spread percentile  : {spread_pctile:.0f} "
      f"(vs {len(df_pctile):,} obs since {SPREAD_PCTILE_START[:4]})")
print(f"  Spread z-score     : {latest['spread_z']:+.2f}σ")
print("=" * 58)

In [ ]:
# ============================================================
# CELL 4 — Three-panel price and spread chart
# ============================================================
cutoff_recent = pd.Timestamp.today() - pd.Timedelta(days=RECENT_DAYS)
df_plot = df_daily[df_daily["date"] >= cutoff_recent].copy()

# Spread has NaN gaps on one-sided sessions — take the last real one
spread_obs    = df_plot[df_plot["spread"].notna()]
last_spread_x = spread_obs["date"].iloc[-1]
last_spread_y = spread_obs["spread"].iloc[-1]
last_z        = spread_obs["spread_z"].dropna().iloc[-1]
last_row      = df_plot.iloc[-1]

# ── Figure setup ─────────────────────────────────────────────
# Margins set explicitly because label_series() needs reserved
# space on the right and tight_layout is off (see finish()).
fig = plt.figure(figsize=(13, 10.5))
gs  = GridSpec(3, 1, figure=fig, hspace=0.42,
               height_ratios=[2.4, 1.5, 1.0],
               left=0.07, right=0.86, top=0.885, bottom=0.07)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# ── Panel 1 — prices ─────────────────────────────────────────
# No 52-week band. A rolling high/low envelope of a series can
# never be crossed by that series, so it says nothing about
# whether today is unusual — it just costs ink.
ax1.plot(df_plot["date"], df_plot["wti"],
         color=cs.COLOR_WTI,   **cs.PRIMARY, zorder=3)
ax1.plot(df_plot["date"], df_plot["brent"],
         color=cs.COLOR_BRENT, **cs.PRIMARY, zorder=3)

ax1.set_title("Front-month price", loc="left")
ax1.set_ylabel("$ / bbl")
ax1.yaxis.set_major_formatter(cs.money(0))

# ── Panel 2 — spread ─────────────────────────────────────────
ax2.fill_between(
    df_plot["date"],
    df_plot["spread_roll_mean"] - 2 * df_plot["spread_roll_std"],
    df_plot["spread_roll_mean"] + 2 * df_plot["spread_roll_std"],
    color=cs.BAND, alpha=0.28, linewidth=0, zorder=1, label="±2σ",
)
ax2.fill_between(
    df_plot["date"],
    df_plot["spread_roll_mean"] - df_plot["spread_roll_std"],
    df_plot["spread_roll_mean"] + df_plot["spread_roll_std"],
    color=cs.BAND, alpha=0.45, linewidth=0, zorder=1, label="±1σ",
)
ax2.plot(df_plot["date"], df_plot["spread_roll_mean"],
         color=cs.GRAY, **cs.DERIVED, zorder=2, label="252d mean")
ax2.plot(df_plot["date"], df_plot["spread"],
         color=cs.COLOR_SPREAD, **cs.PRIMARY, zorder=3)
ax2.axhline(0, color=cs.RULE, linewidth=1.0, zorder=1)

ax2.set_title("Brent–WTI spread", loc="left")
ax2.set_ylabel("$ / bbl")
ax2.yaxis.set_major_formatter(cs.money(0))
ax2.legend(loc="upper left", ncol=3)

# ── Panel 3 — z-score ────────────────────────────────────────
# One color. A bar's position above or below zero already
# encodes its sign; coloring by sign would say it twice.
ax3.bar(df_plot["date"], df_plot["spread_z"],
        color=cs.COLOR_SPREAD, width=2.5, alpha=0.8, zorder=2)
ax3.axhline(0, color=cs.INK_MUTED, linewidth=1.0, zorder=1)
for level in (2, -2):
    ax3.axhline(level, color=cs.RULE, linewidth=0.9,
                linestyle="--", zorder=1)

ax3.set_title("Spread z-score (252-day rolling window)", loc="left")
ax3.set_ylabel("σ")

# ── Shared x-axis ────────────────────────────────────────────
# Same xlim on all three BEFORE labelling: label_series widens
# whatever axes it's given, and panels widened by different
# amounts stop lining up vertically.
x0, x1 = df_plot["date"].min(), df_plot["date"].max()
pad    = (x1 - x0) * 0.01

for ax in (ax1, ax2, ax3):
    ax.set_xlim(x0 - pad, x1 + pad)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    for when, text in EVENTS.items():
        if x0 <= when <= x1:
            cs.event_marker(ax, when, text if ax is ax1 else "")

# ── Right-edge labels: name and value, no leader lines ───────
cs.label_series(ax1, [
    (last_row["date"], last_row["wti"],
     f"WTI  ${last_row['wti']:,.2f}",     cs.COLOR_WTI),
    (last_row["date"], last_row["brent"],
     f"Brent  ${last_row['brent']:,.2f}", cs.COLOR_BRENT),
], gutter=0.18)
cs.label_series(ax2, [
    (last_spread_x, last_spread_y,
     f"${last_spread_y:,.2f}", cs.COLOR_SPREAD),
], gutter=0.18)
cs.label_series(ax3, [
    (last_spread_x, last_z, f"{last_z:+.2f}σ", cs.COLOR_SPREAD),
], gutter=0.18)

# Trim ticks out of the label gutter — that widened space is for
# labels, not a date range with no data in it.
x1_num = mdates.date2num(x1)
for ax in (ax1, ax2, ax3):
    ax.set_xticks([t for t in ax.get_xticks() if t <= x1_num])

# ── Title and footer ─────────────────────────────────────────
fig.suptitle("WTI & Brent crude monitor", x=0.07, y=0.962, ha="left")

latest_date_str = df_daily["date"].max().strftime("%B %d, %Y")
cs.footer(fig, "CME via yfinance", latest_date_str,
          note=f"spread on {n_observed:,} two-sided sessions")

chart_path = (
    CHART_DIR /
    f"wti_brent_monitor_{df_daily['date'].max():%Y-%m-%d}.png"
)
cs.finish(fig, chart_path, tight=False)
plt.show()
print(f"\n  Chart saved → {chart_path}")

In [ ]:
# ============================================================
# CELL 5 — Load inventory signal and price comparison
# ============================================================
# Loads the most recent inventory CSV written by Tool 1,
# aligns it to weekly prices, and measures whether the
# inventory z-score has any relationship with subsequent
# price moves.
# ─────────────────────────────────────────────────────────────

# ── Find most recent Tool 1 CSV ──────────────────────────────
# Sorted by modification time, NOT alphabetically. Filenames
# containing month names sort wrong — "May" lands after "June"
# because M > J. That bug once made this notebook display $90
# WTI against a $69 market.
inventory_files = sorted(
    glob.glob(str(DATA_DIR / "crude_inventory_*.csv")),
    key=lambda f: os.path.getmtime(f)
)

if not inventory_files:
    print("  !! No inventory CSV found in data/ folder.")
    print("  Run Tool 1 (eia_inventory_tracker.ipynb) first,")
    print("  then rerun this cell.")
    raise FileNotFoundError("Inventory data not found.")

latest_inv_file = inventory_files[-1]
print("  Loading inventory data from:")
print(f"  {Path(latest_inv_file).name}")
print()

# ── Load and prepare ─────────────────────────────────────────
df_inv = pd.read_csv(latest_inv_file, parse_dates=["date"])
df_inv = df_inv.sort_values("date").reset_index(drop=True)

df_inv = df_inv.rename(columns={
    "deviation_std":      "us_inv_z",
    "cush_deviation_std": "cush_inv_z",
    "deviation_mmbbl":    "us_dev_mmbbl",
})

# ── Merge with weekly prices ─────────────────────────────────
# merge_asof for nearest-date matching: EIA's week-ending date
# and the market calendar don't always coincide exactly.
df_weekly_sorted = df_weekly.sort_values("date")

df_combined = pd.merge_asof(
    df_weekly_sorted,
    df_inv[["date", "us_inv_z", "cush_inv_z",
            "us_crude_stocks", "us_dev_mmbbl"]],
    on="date",
    direction="nearest",
    tolerance=pd.Timedelta(days=7),
)

df_combined = df_combined.dropna(
    subset=["wti", "us_inv_z"]
).reset_index(drop=True)

print(f"  Combined rows  : {len(df_combined)} weeks with "
      f"both price and inventory data")
print(f"  Date range     : {df_combined['date'].min().date()} → "
      f"{df_combined['date'].max().date()}")

# ── Forward returns ──────────────────────────────────────────
df_combined["wti_fwd1w"] = df_combined["wti"].shift(-1)
df_combined["wti_fwd4w"] = df_combined["wti"].shift(-4)
df_combined["wti_ret1w"] = (
    (df_combined["wti_fwd1w"] - df_combined["wti"])
    / df_combined["wti"] * 100
)
df_combined["wti_ret4w"] = (
    (df_combined["wti_fwd4w"] - df_combined["wti"])
    / df_combined["wti"] * 100
)

# ── Correlation, with an honest error bar ────────────────────
# A bare correlation is uninterpretable without a sample size.
# r = -0.15 on 60 weeks is noise; the same r on 900 weeks is a
# real, if weak, relationship. Fisher's z-transform gives an
# approximate 95% interval:
#     z  = arctanh(r),  SE = 1/sqrt(n-3),
#     CI = tanh(z ± 1.96·SE)
#
# The 4-week series needs one more adjustment. Consecutive rows
# share 3 of their 4 weeks, so the observations aren't
# independent and n overstates the real information content.
# Dividing by the overlap is crude but honest — it is far
# closer to the truth than pretending every row is new.
def corr_with_ci(x, y, overlap=1):
    pair = pd.concat([x, y], axis=1).dropna()
    n_raw = len(pair)
    if n_raw < 10:
        return np.nan, np.nan, np.nan, 0, 0
    r = pair.iloc[:, 0].corr(pair.iloc[:, 1])
    n_eff = max(int(n_raw / overlap), 4)
    z     = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se    = 1 / np.sqrt(n_eff - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    return r, lo, hi, n_raw, n_eff

r1, lo1, hi1, n1, ne1 = corr_with_ci(
    df_combined["us_inv_z"], df_combined["wti_ret1w"], overlap=1)
r4, lo4, hi4, n4, ne4 = corr_with_ci(
    df_combined["us_inv_z"], df_combined["wti_ret4w"], overlap=4)

corr_1w, corr_4w = r1, r4   # kept for Cells 6 and 7

def _verdict(lo, hi):
    if lo < 0 < hi:
        return "interval spans zero — no detectable signal"
    return "interval excludes zero" + (" (negative)" if hi < 0
                                       else " (positive)")

print()
print("=" * 58)
print("  Inventory signal vs forward WTI returns")
print("=" * 58)
print(f"  1-week : r = {r1:+.3f}   95% CI [{lo1:+.3f}, {hi1:+.3f}]")
print(f"           n = {n1}  ({_verdict(lo1, hi1)})")
print()
print(f"  4-week : r = {r4:+.3f}   95% CI [{lo4:+.3f}, {hi4:+.3f}]")
print(f"           n = {n4}, effective n = {ne4} after overlap")
print(f"           ({_verdict(lo4, hi4)})")
print()
print("  Reading it:")
print("  Negative r = below-average inventory precedes rising")
print("  prices — the textbook relationship. But an interval")
print("  that spans zero means this sample cannot distinguish")
print("  the effect from noise, whatever the point estimate.")
print("=" * 58)

In [ ]:
# ============================================================
# CELL 6 — WTI price vs inventory signal
# ============================================================
# Three stacked panels sharing one x-axis:
#   price · inventory z-score · signal summary table
# ─────────────────────────────────────────────────────────────

# ── Staleness check ──────────────────────────────────────────
combined_wti  = df_combined["wti"].iloc[-1]
combined_date = df_combined["date"].iloc[-1].strftime("%b %d")
current_wti   = df_daily["wti"].iloc[-1]
current_date  = df_daily["date"].iloc[-1].strftime("%b %d")

if abs(current_wti - combined_wti) > 5:
    print("!" * 58)
    print("  STALENESS WARNING")
    print(f"  Dashboard WTI : ${combined_wti:.2f} (inventory cutoff {combined_date})")
    print(f"  Current WTI   : ${current_wti:.2f} (cache as of {current_date})")
    print(f"  Gap           : ${abs(current_wti - combined_wti):.2f}")
    print("  Action        : Run Tool 1 first, then rerun Tool 2")
    print("!" * 58)
    print()

# ── Figure ───────────────────────────────────────────────────
fig = plt.figure(figsize=(9.5, 9.6))
gs  = GridSpec(3, 1, figure=fig, hspace=0.45,
               height_ratios=[2.2, 1.2, 1.3],
               left=0.085, right=0.965, top=0.885, bottom=0.05)

ax_price = fig.add_subplot(gs[0])
ax_inv   = fig.add_subplot(gs[1])
ax_table = fig.add_subplot(gs[2])

# ── Panel 1 — WTI price ──────────────────────────────────────
# Two stacked panels, NOT twinx. A dual-axis chart lets the
# scaling manufacture an apparent correlation — slide either
# axis and the relationship appears or vanishes. Shared x with
# separate y keeps the comparison honest, and costs nothing:
# the reader was comparing shapes, not reading both axes.
ax_price.plot(df_combined["date"], df_combined["wti"],
              color=cs.COLOR_WTI, **cs.PRIMARY, zorder=3)
ax_price.set_title("WTI price", loc="left")
ax_price.set_ylabel("$ / bbl")
ax_price.yaxis.set_major_formatter(cs.money(0))

# ── Panel 2 — inventory z-score ──────────────────────────────
ax_inv.bar(df_combined["date"], df_combined["us_inv_z"],
           color=cs.COLOR_INVENTORY, width=4, alpha=0.8, zorder=2)
ax_inv.axhline(0, color=cs.INK_MUTED, linewidth=1.0, zorder=1)
for level in (1, -1):
    ax_inv.axhline(level, color=cs.RULE, linewidth=0.9,
                   linestyle="--", zorder=1)
ax_inv.set_title("US crude inventory vs 5-year seasonal range", loc="left")
ax_inv.set_ylabel("σ")

# ── Shared x-axis ────────────────────────────────────────────
x0, x1 = df_combined["date"].min(), df_combined["date"].max()
pad    = (x1 - x0) * 0.01
for ax in (ax_price, ax_inv):
    ax.set_xlim(x0 - pad, x1 + pad)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    for when, text in EVENTS.items():
        if x0 <= when <= x1:
            cs.event_marker(ax, when, text if ax is ax_price else "")

last = df_combined.iloc[-1]
cs.label_series(ax_price, [
    (last["date"], last["wti"], f"${last['wti']:,.2f}", cs.COLOR_WTI),
], gutter=0.16)
cs.label_series(ax_inv, [
    (last["date"], last["us_inv_z"],
     f"{last['us_inv_z']:+.2f}σ", cs.COLOR_INVENTORY),
], gutter=0.16)

x1_num = mdates.date2num(x1)
for ax in (ax_price, ax_inv):
    ax.set_xticks([t for t in ax.get_xticks() if t <= x1_num])

# ── Panel 3 — signal summary ─────────────────────────────────
ax_table.axis("off")

latest_spread   = df_daily["spread"].dropna().iloc[-1]
latest_spread_z = df_daily["spread_z"].dropna().iloc[-1]

# The verdict defers to the confidence interval, not the point
# estimate. With ~40 weeks of overlap the interval spans zero,
# so the honest reading is "no signal" — not "inconclusive",
# which implies the data had something to say and didn't.
if lo4 < 0 < hi4:
    corr_verdict = f"no signal (n_eff={ne4})"
else:
    corr_verdict = "directionally bullish" if hi4 < 0 else "directionally bearish"

table_data = [
    ["Metric", "Current", "Reading"],
    ["WTI price",            f"${last['wti']:.2f}/bbl",   "—"],
    ["Brent price",          f"${last['brent']:.2f}/bbl", "—"],
    ["Brent–WTI spread",     f"${latest_spread:.2f}/bbl",
     f"{latest_spread_z:+.2f}σ "
     f"({'WIDE' if latest_spread_z > 1 else 'NORMAL'})"],
    ["US inventory z-score", f"{last['us_inv_z']:+.2f}σ",
     "mild BULLISH" if last["us_inv_z"] < -0.5 else "NEUTRAL"],
    ["Inventory vs 5yr avg", f"{last['us_dev_mmbbl']:+.1f} MMbbl",
     "below avg" if last["us_dev_mmbbl"] < 0 else "above avg"],
    ["Inv→price corr (4wk)", f"{corr_4w:+.3f}", corr_verdict],
]

table = ax_table.table(cellText=table_data[1:],
                       colLabels=table_data[0],
                       cellLoc="left", loc="center",
                       bbox=[0, 0, 1, 0.88],
                       colWidths=[0.34, 0.28, 0.38])
table.auto_set_font_size(False)
table.set_fontsize(cs.SIZE_TICK)

for j in range(3):
    table[0, j].set_facecolor(cs.INK)
    table[0, j].set_text_props(color="white", fontweight="semibold")
for i in range(len(table_data)):
    for j in range(3):
        c = table[i, j]
        c.set_edgecolor(cs.RULE)
        c.set_linewidth(0.7)
        c.PAD = 0.04
        if i > 0 and i % 2 == 0:
            c.set_facecolor("#F2F1EE")

ax_table.set_title("Signal summary", loc="left")

# ── Title and footer ─────────────────────────────────────────
fig.suptitle("WTI Price & Inventory Signal", x=0.085, y=0.962, ha="left")
latest_date_str = df_daily["date"].max().strftime("%B %d, %Y")
cs.footer(fig, "EIA API + CME via yfinance", latest_date_str,
          note=f"{len(df_combined)} weeks of overlap")

chart_path = (
    CHART_DIR /
    f"price_inventory_overlay_{df_daily['date'].max():%Y-%m-%d}.png"
)
cs.finish(fig, chart_path, tight=False)
plt.show()
print(f"\n  Chart saved → {chart_path}")

In [ ]:
# ============================================================
# CELL 7 — Save data and commentary log
# ============================================================

csv_filename = (
    f"price_inventory_combined_"
    f"{df_daily['date'].max():%Y-%m-%d}.csv"
)
csv_path = DATA_DIR / csv_filename

save_cols = [
    "date", "wti", "brent", "spread",
    "spread_roll_mean", "spread_roll_std", "spread_z",
    "us_inv_z", "us_crude_stocks", "us_dev_mmbbl",
]
df_combined[[c for c in save_cols
             if c in df_combined.columns]].to_csv(csv_path, index=False)
print(f"  Data saved → {csv_path}")

# ── Commentary log ───────────────────────────────────────────
commentary_path = DATA_DIR / "wti_brent_commentary_log.txt"

last            = df_combined.iloc[-1]
latest_spread   = df_daily["spread"].dropna().iloc[-1]
latest_spread_z = df_daily["spread_z"].dropna().iloc[-1]
latest_wti_p    = df_daily["wti"].iloc[-1]
latest_brent_p  = df_daily["brent"].iloc[-1]

spread_signal = (
    "WIDE — geopolitical/logistical premium"  if latest_spread_z >  1.5 else
    "ELEVATED — above historical norm"        if latest_spread_z >  0.5 else
    "NORMAL — within historical range"        if latest_spread_z > -0.5 else
    "NARROW — WTI at premium to historical norm"
)

inv_signal = (
    "BULLISH" if last["us_inv_z"] < -0.5
    else "NEUTRAL" if abs(last["us_inv_z"]) <= 0.5
    else "BEARISH"
)

# The correlation gets reported with its interval, or not at all.
# A bare r in a log that accumulates week after week is exactly
# how a number with no power turns into a remembered "finding".
if lo4 < 0 < hi4:
    corr_line = (
        f"r = {corr_4w:+.3f}, 95% CI [{lo4:+.3f}, {hi4:+.3f}] — "
        f"spans zero, no detectable signal (n_eff = {ne4})"
    )
else:
    direction = "bullish" if hi4 < 0 else "bearish"
    corr_line = (
        f"r = {corr_4w:+.3f}, 95% CI [{lo4:+.3f}, {hi4:+.3f}] — "
        f"directionally {direction} (n_eff = {ne4})"
    )

commentary_entry = f"""
{'=' * 60}
WTI & BRENT PRICE MONITOR COMMENTARY
Run date  : {dt.date.today():%B %d, %Y}
Data date : {latest_date_str}
{'=' * 60}

METRICS SNAPSHOT
  WTI price           : ${latest_wti_p:.2f}/bbl
  Brent price         : ${latest_brent_p:.2f}/bbl
  Brent-WTI spread    : ${latest_spread:.2f}/bbl
  Spread z-score      : {latest_spread_z:+.2f} sigma
  Spread signal       : {spread_signal}
  Spread percentile   : {spread_pctile:.0f} (since {SPREAD_PCTILE_START[:4]},
                        {len(df_pctile):,} obs)

  US inventory z-score: {last['us_inv_z']:+.2f} sigma
  US stocks vs 5yr avg: {last['us_dev_mmbbl']:+.1f} MMbbl
  Inventory signal    : {inv_signal}

  Inv->price, 4 weeks : {corr_line}
  Overlap window      : {len(df_combined)} weeks
                        ({df_combined['date'].min():%Y-%m-%d} to
                         {df_combined['date'].max():%Y-%m-%d})

DATA QUALITY
  Spread computed on {n_observed:,} of {n_total:,} sessions where both
  legs traded; {n_gap:,} one-sided sessions left blank rather than
  forward-filled.

PLAIN ENGLISH SUMMARY
  As of {latest_date_str}, WTI traded at ${latest_wti_p:.2f}/bbl and
  Brent at ${latest_brent_p:.2f}/bbl. The Brent-WTI spread stood at
  ${latest_spread:.2f}/bbl ({latest_spread_z:+.2f} sigma vs its 252-day
  rolling mean) — {spread_signal.lower()}.

  US crude inventory was {last['us_inv_z']:+.2f} sigma against its
  five-year seasonal range, with stocks
  {abs(last['us_dev_mmbbl']):.1f} MMbbl
  {'below' if last['us_dev_mmbbl'] < 0 else 'above'} the five-year average.

  The inventory-to-price relationship cannot be assessed from this
  sample: Tool 1 exports only the current year, giving {len(df_combined)}
  overlapping weeks and an effective sample of {ne4} after accounting
  for the 4-week return overlap. Treat the correlation as absent
  rather than weak until Tool 1 exports a rolling-z history.

"""

with open(commentary_path, "a", encoding="utf-8") as f:
    f.write(commentary_entry)

print(f"  Commentary logged → {commentary_path.name}")

# ── Next refresh ─────────────────────────────────────────────
today          = dt.date.today()
days_to_wed    = (2 - today.weekday()) % 7
next_wednesday = today + dt.timedelta(days=days_to_wed)

print()
print("=" * 58)
print("  TOOL 2 COMPLETE")
print("=" * 58)
print(f"""
  Built this run
  ──────────────────────────────────────────────────
  · {len(df_daily):,} days of WTI + Brent history
  · Spread on {n_observed:,} two-sided sessions
  · Spread z-score vs 252-day rolling window
  · Three-panel price/spread/signal chart
  · Inventory overlay, {len(df_combined)} weeks
  · Combined dataset + commentary appended

  To refresh
  ──────────────────────────────────────────────────
  1. Kernel → Restart and Run All
  2. Cache tops up with new trading days only
  3. Next EIA release: Wednesday {next_wednesday}, 10:30am ET
  4. Run Tool 1 first on Wednesdays, then Tool 2

  Watch
  ──────────────────────────────────────────────────
  · Spread z normalizing toward 0 — Hormuz risk premium
    being priced out
  · Spread holding above +2 sigma — market still pricing
    sustained disruption
  · Inventory z turning positive — supply rebuilding
""")